<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Normalise before you compare

**[Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/)** · Speech AI: Transcribe, Understand and Speak · Module 2, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** apply the course's normalisation guide to a reference and a model transcript, see how much the word error rate (WER) changes when only the normalisation changes, find the mistake of normalising one side only, and test one new rule on the French calls.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | How to count substitutions, deletions and insertions, from [Word error rate, step by step](https://learning.nextia-ai.com/courses/speech/m02/word-error-rate-step-by-step/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M02-L02-normalise-before-you-compare/normalise-before-you-compare-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "data/recordings/asr/vosk-small.jsonl": "cfb072991ed3c2de1a9604b1f001952de061a4231999b4051f2b20f93b7641be",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The same transcript, three scores

The caller said "Yes, it's 742155." (an order number, said digit by digit). The model wrote every word right, in words: "yes it is seven four two one five five". The course's helper `cp.normalise(text, level, language)` has three levels:

| Level | What it does |
|---|---|
| `none` | splits the text on spaces, nothing else |
| `basic` | lower case, punctuation removed |
| `full` | basic, plus the rules of the guide: numbers, contractions, fillers, French elisions, ordinals |

> **Predict.** Only the normalisation changes in the next cell. Can the WER go from over 100% to 0%?

In [ ]:
reference_text = "Yes, it's 742155."
model_text = "yes it is seven four two one five five"
for level in ["none", "basic", "full"]:
    r = cp.normalise(reference_text, level, "en")
    h = cp.normalise(model_text, level, "en")
    print(f"{level:5}  reference: {r!r:24} model: {h!r:42} WER {cp.wer(r, h):.1%}")

> **Check.** `none` gives 300.0%, `basic` gives 266.7% and `full` gives 0.0%. The model made no mistake: the first two scores measure the way the numbers are written, not what the model heard.

## 2. The guide, rule by rule

The full level follows a written guide. Print it with `print(cp.NORMALISATION_GUIDE)`. Run the next cell to see what `basic` and `full` do to six short texts.

In [ ]:
print(cp.NORMALISATION_GUIDE)

In [ ]:
examples = [
    ("en", "Um, it's seventy-four, twenty-one, fifty-five."),
    ("en", "Yes, it's 38.45.29."),
    ("en", "It shipped on the fourteenth of March."),
    ("en", "Seven oh two, and I don't want it."),
    ("fr", "Euh, c'est le sept quatre deux un cinq cinq."),
    ("fr", "Elle est partie le premier juillet."),
]
for language, text in examples:
    print(text)
    print("   basic:", cp.normalise(text, "basic", language))
    print("   full: ", cp.normalise(text, "full", language))

> **Check.** Look at each line. `full` turns "seventy-four, twenty-one, fifty-five" and "38.45.29" into the same `384529`-style number, removes "um" and "euh", writes "it's" as "it is" and "don't" as "do not", splits "c'est" into "c est", and turns "the fourteenth" and "le premier" into numbers. `basic` leaves "38 45 29" as three words, so it would count two insertions.

## 3. The whole test set at three levels

Now the 60 test calls: the reference transcripts, and the recorded transcripts of two recognisers on the phone audio. Only the normalisation level changes.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]

def recorded(method):
    """The recorded transcripts of one recogniser on the phone audio: {call_id: text}."""
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{method}.jsonl", encoding="utf-8")]
    return {r["call_id"]: r["text"] for r in rows if r["kind"] == "phone"}

small = recorded("whisper-small")
vosk = recorded("vosk-small")
print(len(test), "test calls;", len(small), "and", len(vosk), "recorded transcripts")

In [ ]:
scores = {}
for name, texts in [("Whisper small", small), ("Vosk small", vosk)]:
    for level in ["none", "basic", "full"]:
        s = cp.corpus_scores([(reference(c), texts[c["call_id"]], c["language"]) for c in test], level)
        scores[(name, level)] = s["wer"]
        print(f"{name:14} {level:5}  reference words {s['words']:5}  WER {s['wer']:.1%}")

> **Check.** Whisper small: 18.2%, 11.5%, 8.6%. Vosk small: 60.2%, 35.5%, 31.6%. The recognisers did not change; only the way we compare the texts did. Vosk writes every number in words and has no punctuation, so the guide changes its score most. Notice also that the number of reference words changes: "it's" is one word before the guide and two after it.

## 4. Failure case: normalise one side only

The guide must go to **both** texts. Here the reference stays as it was written, and only the model text is normalised.

In [ ]:
errors = words = 0
for c in test:
    r = cp.normalise(reference(c), "none", c["language"])     # the reference as it was written
    h = cp.normalise(small[c["call_id"]], "full", c["language"])  # only the model text normalised
    n = cp.counts(r, h)
    errors += n["sub"] + n["del"] + n["ins"]
    words += n["ref_words"]
print(f"Whisper small, only the model text normalised: WER {errors / words:.1%}")

> **Check.** 53.3%, against 8.6% with the guide on both sides. Capital letters, punctuation and digits in the reference now never match the model text. **Fix:** always normalise the reference and the model transcript with the same function and the same level, as `cp.corpus_scores` does.

## 5. Your turn: one more rule for French

Whisper small sometimes writes a French ordinal as "1er" ("le 1er juillet"). The guide turns "premier" into 1, but it has no rule for "1er", so "1" and "1er" count as a substitution.

> **Your turn.** Write `guide_plus(text, language)`: the full guide, then one more rule that turns a number followed by `er`, `re`, `e` or `ème` into the number alone (a regular expression with `re.sub` is enough). Then compute the WER of Whisper small on the French test calls with the guide (`before`) and with your new rule (`after`). Normalise **both** texts with the same function.

> **Predict.** Will the French WER go down by more or less than one point?

In [ ]:
import re

def guide_plus(text, language):
    out = cp.normalise(text, "full", language)
    return ...  # add the rule here

french = [c for c in test if c["language"] == "fr"]
before = ...
after = ...

In [ ]:
expect('before', before, 0.1506, tolerance=0.001)

In [ ]:
expect('after', after, 0.15, tolerance=0.001)

> **Check.** 15.06% before and 15.00% after. The rule is right, and the change is small: most of the French errors are real recognition errors. A change to the guide changes every score, so the course keeps one fixed guide and scores every recogniser with it. If you change your guide, score every recogniser again and write the change down.

## Next

You can now explain why a WER means nothing without its normalisation, and apply the guide to both sides. The next lesson counts the errors that matter to Larkfield (order numbers, product names, negations) apart from the WER. The [lesson page](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/) has the explorer and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The errors that matter](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.